In [6]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\zen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\zen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\zen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [8]:
# HELPER FUNCTIONS - Provided for quantum mechanics

def generate_quantum_random_bits(n):
    """Generate n random bits using quantum measurements (not Python random)."""
    bits = []
    for i in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)  # Superposition: (|0⟩ + |1⟩)/√2
        qc.measure(0, 0)
        
        simulator = BasicSimulator()
        job = simulator.run(qc, shots=1)
        result = job.result()
        counts = result.get_counts()
        bit = int(list(counts.keys())[0])
        bits.append(bit)
    
    return bits


def encode_qubit(bit, basis):
    """Encode a classical bit into a quantum state.
    
    basis: 0 = rectilinear (+), 1 = diagonal (x)
    Returns a Statevector representing the encoded state
    """
    sqrt2 = 2**0.5
    
    if basis == 0:  # Rectilinear basis (+)
        if bit == 0:
            return Statevector([1, 0])  # |0⟩
        else:
            return Statevector([0, 1])  # |1⟩
    else:  #diagonal basis (x)
        if bit == 0:
            return Statevector([1/sqrt2, 1/sqrt2])  # |+⟩
        else:
            return Statevector([1/sqrt2, -1/sqrt2])  # |-⟩


def measure_qubit_quantum(state, basis):
    """Measure a quantum state using a quantum circuit (not PRNG).
    
    state: Statevector to measure
    basis: 0 = rectilinear (+), 1 = diagonal (x)
    Returns the measurement result (0 or 1)
    """
    sqrt2 = 2**0.5
    
    # Convert statevector to circuit by determining which state it is
    qc = QuantumCircuit(1, 1)
    
    # Determine which state we have and rebuild it with gates
    state_array = state.data
    
    # Check if it's |0⟩
    if math.isclose(abs(state_array[0]), 1) and math.isclose(abs(state_array[1]), 0):
        qc.id(0)  # |0⟩ - do nothing
    # Check if it's |1⟩
    elif math.isclose(abs(state_array[0]), 0) and math.isclose(abs(state_array[1]), 1):
        qc.x(0)  # |1⟩
    # Check if it's |+⟩ (both components positive)
    elif math.isclose(state_array[0].real, 1/sqrt2) and math.isclose(state_array[1].real, 1/sqrt2):
        qc.h(0)  # |+⟩
    # Check if it's |-⟩ (first positive, second negative)
    elif math.isclose(state_array[0].real, 1/sqrt2) and math.isclose(state_array[1].real, -1/sqrt2):
        qc.h(0)  # First create |+⟩
        qc.z(0)  # Then apply Z to get |-⟩
    
    # Apply basis rotation if measuring in diagonal basis
    if basis == 1:  # Diagonal basis
        qc.h(0)  # Rotate to computational basis for measurement
    
    # Measure
    qc.measure(0, 0)
    
    # Run on simulator to get measurement outcome
    simulator = BasicSimulator()
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    bit = int(list(counts.keys())[0])
    
    return bit


print("BB84 Protocol - Plain (No Attacker)")
print("=" * 50)

BB84 Protocol - Plain (No Attacker)


In [9]:
print("\n1. ALICE generates random bits and bases")
print("-" * 50)

n_qubits = 100

#generate Alice's random bits using quantum measurements
alice_bits = generate_quantum_random_bits(n_qubits)
print(f"Alice's bits (first 20): {alice_bits[:20]}")

#generate Alice's random bases using quantum measurements
alice_bases = generate_quantum_random_bits(n_qubits)
print(f"Alice's bases (first 20, 0=+, 1=x): {alice_bases[:20]}")

#encode each bit using the corresponding basis
alice_qubits = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(n_qubits)]
print(f"Alice has prepared {len(alice_qubits)} qubits")


1. ALICE generates random bits and bases
--------------------------------------------------
Alice's bits (first 20): [1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0]
Alice's bases (first 20, 0=+, 1=x): [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1]
Alice has prepared 100 qubits


In [10]:
# STEP 2: BOB - Receive qubits and measure
print("\n2. BOB receives qubits and measures")
print("-" * 50)

#generate Bob's random bases using quantum measurements
bob_bases = generate_quantum_random_bits(n_qubits)
print(f"Bob's bases (first 20, 0=+, 1=x): {bob_bases[:20]}")

#for each qubit received, measure it with Bob's randomly chosen basis
bob_measurements = [measure_qubit_quantum(alice_qubits[i], bob_bases[i]) for i in range(n_qubits)]
print(f"Bob's measurements (first 20): {bob_measurements[:20]}")

# STEP 3: SIFT THE KEY - Compare bases publicly
print("\n3. SIFTING: Public basis comparison")
print("-" * 50)

#keep only bits where Alice and Bob used the same basis
sifted_key_alice = [alice_bits[i] for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
sifted_key_bob = [bob_measurements[i] for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]

print(f"Sifted key length: {len(sifted_key_alice)} bits")
print(f"Sifting efficiency: {len(sifted_key_alice)/n_qubits*100:.1f}%")

print(f"\nAlice's sifted key (first 20): {sifted_key_alice[:20]}")
print(f"Bob's sifted key (first 20): {sifted_key_bob[:20]}")


# STEP 4: CHECK FOR EAVESDROPPING
print("\n4. EAVESDROPPING CHECK")
print("-" * 50)

#compare a random subset of the sifted key to check for eavesdropping
n_check_target = min(len(sifted_key_alice) // 2, 20)
mask_bits = generate_quantum_random_bits(len(sifted_key_alice))
check_indices = [i for i, bit in enumerate(mask_bits) if bit == 1]
if not check_indices:
    check_indices = list(range(n_check_target))
n_check = min(n_check_target, len(check_indices))
check_indices = check_indices[:n_check]

#count bit mismatches
errors = sum(1 for idx in check_indices if sifted_key_alice[idx] != sifted_key_bob[idx])

error_rate = errors / n_check if n_check > 0 else 0
print(f"Bits checked: {n_check}")
print(f"Errors detected: {errors}")
print(f"Quantum Bit Error Rate (QBER): {error_rate*100:.1f}%")
print(f"Expected error rate (no eavesdropping): ~0%")

#set threshold and check if QBER exceeds it
THRESHOLD = 0.05  # 5% threshold

if error_rate > THRESHOLD:
    print(f"\n✗ EAVESDROPPING DETECTED! Error rate {error_rate*100:.1f}% > threshold {THRESHOLD*100:.1f}%")
else:
    print(f"\n✓ Channel appears secure (error rate {error_rate*100:.1f}%)")

# Remove checked bits from sifted key to create final secret key
checked_set = set(check_indices)
final_key = [sifted_key_alice[i] for i in range(len(sifted_key_alice)) if i not in checked_set]

print(f"\nFinal shared secret key length: {len(final_key)} bits")
print(f"Final key (first 20): {final_key[:20]}")


2. BOB receives qubits and measures
--------------------------------------------------
Bob's bases (first 20, 0=+, 1=x): [0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1]
Bob's measurements (first 20): [1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]

3. SIFTING: Public basis comparison
--------------------------------------------------
Sifted key length: 57 bits
Sifting efficiency: 57.0%

Alice's sifted key (first 20): [1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1]
Bob's sifted key (first 20): [1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1]

4. EAVESDROPPING CHECK
--------------------------------------------------
Bits checked: 20
Errors detected: 0
Quantum Bit Error Rate (QBER): 0.0%
Expected error rate (no eavesdropping): ~0%

✓ Channel appears secure (error rate 0.0%)

Final shared secret key length: 37 bits
Final key (first 20): [1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1]
